# Features com informação do passado: risco de furto, roubo e crimes (furto + roubo) por célula H3 × quinzena

**Regra central:** toda feature da quinzena `t` usa apenas dados até `t-1`. Nada da própria quinzena entra.

**Início da série (2022):** não existe informação antes de jan/2022, então:
- lags e médias longas ficam `NaN` ou parciais nas primeiras quinzenas;
- cada linha recebe `n_hist` (quantas quinzenas passadas existem). Esta coluna serve para **filtrar** o treino, **não** deve ser usada como feature (viraria um "ano" disfarçado, e a RF não extrapola);
- as janelas móveis usam `min_periods=1` (média do que existir), então nas primeiras linhas elas são médias curtas. Use `MIN_HIST_TREINO` na seção final para descartar essas linhas.

**Grupos de features geradas**
0. As mesmas features são geradas para **furtos** e **roubos** (conjunto completo) e para o **total** (`n_total`, furto + roubo), todos com o mesmo conjunto completo. Assim o mesmo arquivo serve ao modelo de furto e ao de roubo
1. Lags da própria célula (1 a 24 quinzenas, inclusive mesma quinzena do ano anterior)
2. Médias, máximos e desvio em janelas móveis; médias exponenciais
3. Histórico acumulado (média, fração de quinzenas com crime, tempo desde o último crime)
4. Sazonalidade: média da mesma quinzena do ano em anos anteriores
5. Tendência: razão e diferença entre curto e longo prazo
6. Vizinhança H3 (anel 1 e anel 2), com lag e janelas
7. Nível global (todas as células) no passado, por série
8. Calendário (mês, quinzena, seno/cosseno). Sem ano.

In [1]:
import warnings
import json
import numpy as np
import pandas as pd
from scipy import sparse

try:
    import h3
except ImportError:
    h3 = None

# ---------------- CONFIGURAÇÃO ----------------
PANEL_PATH = "datasets/database_processed_h3.csv"      # ignorado se já existir a variável `panel` na memória
COL_H3, COL_DATA = "h3", "DATA"
TARGET = "n_furtos"               # alvo do teste rápido no fim (use "n_roubos" para avaliar o modelo de roubo)
SERIES = {                        # série -> conjunto de features ("completa" ou "basica")
    "n_furtos": "completa",
    "n_roubos": "completa",
    "n_total": "completa",        # furto + roubo
}

LAGS = [1, 2, 3, 4, 6, 8, 12, 24]   # em quinzenas (24 = mesma quinzena do ano anterior)
JANELAS = [2, 4, 8, 12, 24]         # janelas móveis (quinzenas)
EWM_SPANS = [2, 6, 12]
SAZ_ANOS = 4                        # quantos anos anteriores entram na média sazonal
SUAVIZACAO = 0.1                    # evita divisão por zero nas razões de tendência

USAR_VIZINHOS = True                # precisa do pacote h3
K_VIZ = [1, 2]                      # raios da vizinhança (anel 1 e 2)

MIN_CRIMES_CELULA = 0               # descarta células com menos crimes no total (reduz memória). 0 = mantém todas
OUT_PARQUET = "datasets/features.parquet"

In [2]:
if "panel" not in globals():
    panel = pd.read_csv(PANEL_PATH)

df = panel[[COL_H3, COL_DATA] + list(SERIES)].copy()
df[COL_DATA] = pd.to_datetime(df[COL_DATA])
df[COL_H3] = df[COL_H3].astype(str)

assert not df.duplicated([COL_H3, COL_DATA]).any(), "há (h3, DATA) repetido"
assert df[COL_DATA].dt.day.isin([1, 16]).all(), "DATA deve ser dia 1 ou 16"

datas = np.sort(df[COL_DATA].unique())
cells = np.array(sorted(df[COL_H3].unique()))
T, C = len(datas), len(cells)

d0, d1 = pd.Timestamp(datas[0]), pd.Timestamp(datas[-1])
meses = (d1.year - d0.year) * 12 + d1.month - d0.month + 1
assert T == 2 * meses, f"faltam quinzenas no meio da série: {T} vs {2 * meses}"
assert len(df) == T * C, "grade incompleta: faltam combinações célula × quinzena (preencha com zero antes)"

print(f"{C:,} células × {T} quinzenas = {T * C:,} linhas")
print("primeira quinzena:", d0.date(), "| última:", d1.date())

1,268 células × 108 quinzenas = 136,944 linhas
primeira quinzena: 2022-01-01 | última: 2026-06-16


## Funções auxiliares

Trabalhamos com matrizes `quinzenas × células`. Cada coluna é uma célula, então `shift`, `rolling` e `ewm` agem sempre dentro da própria célula e rodam rápido.

In [3]:
def pivot(col):
    return (df.pivot(index=COL_DATA, columns=COL_H3, values=col)
              .reindex(index=pd.DatetimeIndex(datas), columns=cells)
              .astype("float32"))

# Matrizes (T × C), uma por série
MATS = {c: pivot(c) for c in SERIES}
F = MATS[TARGET]

# Filtro opcional de células (aplicado só ao montar a tabela final; vizinhança usa todas)
ref = MATS["n_total"] if "n_total" in MATS else sum(MATS.values())
manter = (ref.sum(axis=0) >= MIN_CRIMES_CELULA).to_numpy()
cells_k = cells[manter]
Ck = len(cells_k)
print(f"células mantidas: {Ck:,} de {C:,}")

feat = {}   # nome -> vetor achatado (tempo-major), float32


def add(nome, M):
    """Achata matriz (T × C) para vetor e guarda."""
    a = np.asarray(M, dtype="float32")
    if not manter.all():
        a = a[:, manter]
    feat[nome] = a.ravel()


def add_tempo(nome, v):
    """Feature que só depende do tempo (igual para todas as células)."""
    feat[nome] = np.repeat(np.asarray(v, dtype="float32"), Ck)


def feats_basicas(M, p):
    """Conjunto enxuto: usado para o total e para a vizinhança."""
    S = M.shift(1)
    add(f"{p}_lag1", M.shift(1))
    add(f"{p}_lag2", M.shift(2))
    add(f"{p}_mean4", S.rolling(4, min_periods=1).mean())
    add(f"{p}_mean12", S.rolling(12, min_periods=1).mean())
    add(f"{p}_exp_mean", S.expanding(min_periods=1).mean())


def feats_completas(M, p):
    """Conjunto completo (usado em furtos, roubos e total)."""
    S = M.shift(1)                       # S[t] = valor em t-1

    # 1) lags
    for k in LAGS:
        add(f"{p}_lag{k}", M.shift(k))

    # 2) janelas móveis (só passado, via S)
    for w in JANELAS:
        add(f"{p}_mean{w}", S.rolling(w, min_periods=1).mean())
    for w in [4, 12, 24]:
        add(f"{p}_max{w}", S.rolling(w, min_periods=1).max())
    add(f"{p}_std12", S.rolling(12, min_periods=2).std())
    for sp in EWM_SPANS:
        add(f"{p}_ewm{sp}", S.ewm(span=sp, adjust=False).mean())

    # 3) histórico acumulado
    add(f"{p}_exp_mean", S.expanding(min_periods=1).mean())
    teve = (M > 0).astype("float32")
    add(f"{p}_exp_frac_crime", teve.shift(1).expanding(min_periods=1).mean())

    idx = np.arange(T, dtype="float32")[:, None]
    ult = pd.DataFrame(np.where(M.to_numpy() > 0, idx, np.nan),
                       index=M.index, columns=M.columns).ffill().shift(1).to_numpy()
    desde = idx - ult
    desde = np.where(np.isnan(desde), idx + 1, desde)   # nunca teve crime no passado: usa o tamanho do histórico + 1
    add(f"{p}_qz_desde_ultimo", desde)

    # 4) sazonalidade: média da mesma quinzena do ano em anos anteriores
    arr = np.stack([M.shift(24 * k).to_numpy() for k in range(1, SAZ_ANOS + 1)])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        saz = np.nanmean(arr, axis=0)
    add(f"{p}_saz_mean", saz)

    # 5) tendência (curto x longo prazo)
    m4 = S.rolling(4, min_periods=1).mean()
    m12 = S.rolling(12, min_periods=1).mean()
    m24 = S.rolling(24, min_periods=1).mean()
    add(f"{p}_tend_4_24", (m4 + SUAVIZACAO) / (m24 + SUAVIZACAO))
    add(f"{p}_dif_4_12", m4 - m12)
    add(f"{p}_tend_12_exp", (m12 + SUAVIZACAO) / (S.expanding(min_periods=1).mean() + SUAVIZACAO))

células mantidas: 1,268 de 1,268


## 1 a 5. Features da própria célula (furtos, roubos e total)

In [4]:
p_alvo = TARGET.replace("n_", "")
for col, tipo in SERIES.items():
    p = col.replace("n_", "")
    (feats_completas if tipo == "completa" else feats_basicas)(MATS[col], p)
print(len(feat), "features até agora")

81 features até agora


## 6. Vizinhança H3

Soma dos vizinhos dentro do raio, dividida pelo número teórico de vizinhos (células sem registro contam como zero). Tudo com lag, então só passado.

In [5]:
def matriz_vizinhanca(cells, k):
    pos = {c: i for i, c in enumerate(cells)}
    grid_disk = getattr(h3, "grid_disk", None) or getattr(h3, "k_ring")   # h3 v4 ou v3
    r, c_ = [], []
    for c, i in pos.items():
        for v in grid_disk(c, k):
            j = pos.get(v)
            if j is not None and j != i:
                r.append(i)
                c_.append(j)
    return sparse.csr_matrix((np.ones(len(r), dtype="float32"), (r, c_)), shape=(len(cells), len(cells)))


if USAR_VIZINHOS:
    if h3 is None:
        raise ImportError("instale o pacote h3 (pip install h3) ou defina USAR_VIZINHOS = False")
    for k in K_VIZ:
        A = matriz_vizinhanca(cells, k)
        n_teorico = 3 * k * (k + 1)                       # 6 (k=1), 18 (k=2)
        for col in SERIES:
            NB = (A @ MATS[col].to_numpy().T).T / n_teorico   # T × C
            NB = pd.DataFrame(NB, index=F.index, columns=F.columns)
            feats_basicas(NB, f"viz{k}_{col.replace('n_', '')}")
    print(len(feat), "features até agora")

111 features até agora


## 7. Nível global (todas as células), por série

In [6]:
for col in SERIES:
    p = col.replace("n_", "")
    g = pd.Series(MATS[col].sum(axis=1).to_numpy(), index=F.index)
    gS = g.shift(1)
    add_tempo(f"global_{p}_lag1", g.shift(1))
    add_tempo(f"global_{p}_mean4", gS.rolling(4, min_periods=1).mean())
    add_tempo(f"global_{p}_mean12", gS.rolling(12, min_periods=1).mean())
    add_tempo(f"global_{p}_tend_4_12", (gS.rolling(4, min_periods=1).mean() + 1) / (gS.rolling(12, min_periods=1).mean() + 1))
print(len(feat), "features até agora")

123 features até agora


## 8. Calendário

Sem `ano` e sem contador de tempo (a RF não extrapola, como discutimos).

In [7]:
dt = pd.DatetimeIndex(datas)
mes = dt.month.to_numpy()
quinz = np.where(dt.day.to_numpy() > 15, 2, 1)
qz_ano = (mes - 1) * 2 + quinz                              # 1..24

add_tempo("mes", mes)
add_tempo("quinzena", quinz)
add_tempo("qz_ano", qz_ano)
add_tempo("qz_sin", np.sin(2 * np.pi * qz_ano / 24))
add_tempo("qz_cos", np.cos(2 * np.pi * qz_ano / 24))

FEATURES = list(feat.keys())
print(len(FEATURES), "features no total")

128 features no total


## Montagem da tabela final

In [8]:
base = {
    COL_H3: pd.Categorical.from_codes(np.tile(np.arange(Ck), T), categories=cells_k),
    COL_DATA: np.repeat(datas, Ck),
    **{c: MATS[c].to_numpy()[:, manter].astype("float32").ravel() for c in SERIES},   # contagens (para montar seus alvos)
    "n_hist": np.repeat(np.arange(T, dtype="int16"), Ck),    # quinzenas passadas disponíveis (NÃO usar como feature)
}
df_final = pd.DataFrame({**base, **feat})
del feat
print(df_final.shape, f"| {df_final.memory_usage(deep=True).sum() / 1e9:.2f} GB")
df_final.head()

(136944, 134) | 0.07 GB


,h3,DATA,n_furtos,n_roubos,n_total,n_hist,furtos_lag1,furtos_lag2,furtos_lag3,furtos_lag4,...,global_roubos_tend_4_12,global_total_lag1,global_total_mean4,global_total_mean12,global_total_tend_4_12,mes,quinzena,qz_ano,qz_sin,qz_cos
0,8781219aaffffff,2022-01-01,0.0,0.0,0.0,0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,0.258819,0.965926
1,87812e672ffffff,2022-01-01,0.0,0.0,0.0,0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,0.258819,0.965926
2,87a8022d2ffffff,2022-01-01,0.0,0.0,0.0,0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,0.258819,0.965926
3,87a806495ffffff,2022-01-01,0.0,0.0,0.0,0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,0.258819,0.965926
4,87a806c14ffffff,2022-01-01,0.0,0.0,0.0,0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,0.258819,0.965926


## Verificações de vazamento e de início da série

In [9]:
chk = df_final.sort_values([COL_H3, COL_DATA], kind="stable").reset_index(drop=True)
g_ = chk.groupby(COL_H3, observed=True)[TARGET]

# lag1 tem que ser exatamente o valor da quinzena anterior
assert np.allclose(chk[f"{p_alvo}_lag1"].fillna(-1), g_.shift(1).fillna(-1)), "lag1 incorreto"
assert np.allclose(chk[f"{p_alvo}_lag24"].fillna(-1), g_.shift(24).fillna(-1)), "lag24 incorreto"

# média móvel 4 recalculada à mão para uma célula
c0 = chk[COL_H3].iloc[len(chk) // 2]
m = chk[COL_H3] == c0
manual = chk.loc[m, TARGET].shift(1).rolling(4, min_periods=1).mean()
assert np.allclose(manual.fillna(-1), chk.loc[m, f"{p_alvo}_mean4"].fillna(-1), atol=1e-3), "mean4 incorreta"

# nenhuma feature pode ser idêntica ao alvo da mesma linha
for f in FEATURES:
    assert not np.array_equal(chk[f].fillna(-9).to_numpy(), chk[TARGET].to_numpy()) or chk[TARGET].nunique() == 1, f"{f} igual ao alvo"
print("OK: sem vazamento nos testes acima")
del chk

# % de NaN nas primeiras quinzenas (efeito do início da série)
amostra_f = [f"{p_alvo}_lag1", f"{p_alvo}_lag12", f"{p_alvo}_lag24", f"{p_alvo}_saz_mean", f"{p_alvo}_std12"]
ini = df_final[df_final["n_hist"] < 26]
nan_ini = ini.groupby("n_hist")[amostra_f].apply(lambda x: x.isna().mean())
nan_ini.index = [str(pd.Timestamp(datas[i]).date()) for i in nan_ini.index]
nan_ini.round(2)

OK: sem vazamento nos testes acima


,furtos_lag1,furtos_lag12,furtos_lag24,furtos_saz_mean,furtos_std12
2022-01-01,1.0,1.0,1.0,1.0,1.0
2022-01-16,0.0,1.0,1.0,1.0,1.0
2022-02-01,0.0,1.0,1.0,1.0,0.0
2022-02-16,0.0,1.0,1.0,1.0,0.0
2022-03-01,0.0,1.0,1.0,1.0,0.0
2022-03-16,0.0,1.0,1.0,1.0,0.0
2022-04-01,0.0,1.0,1.0,1.0,0.0
2022-04-16,0.0,1.0,1.0,1.0,0.0
2022-05-01,0.0,1.0,1.0,1.0,0.0
2022-05-16,0.0,1.0,1.0,1.0,0.0


Leitura da tabela: `lag24` e `saz_mean` só ficam completos a partir da 25ª quinzena (jan/2024). Antes disso são `NaN`. A RF do scikit-learn aceita `NaN` (versão 1.4+); em outras versões preencha com `-1`.

Decisão sobre o início da série (veja `MIN_HIST_TREINO` abaixo):
- `0`: usa tudo, mas as features longas são parciais no começo;
- `12`: descarta o 1º ano (sem 12 quinzenas de passado), mantendo lags até 12 completos;
- `24`: descarta 2022 e 2023, todas as features completas (inclusive sazonalidade), porém treino bem menor.

## Salvar

In [10]:
try:
    df_final.to_parquet(OUT_PARQUET, index=False)
    print("salvo em", OUT_PARQUET)
except Exception as e:
    print("parquet indisponível:", e)
    df_final.to_pickle(OUT_PARQUET.replace(".parquet", ".pkl"))

with open("features_lista.json", "w") as fh:
    json.dump(FEATURES, fh, indent=1)

salvo em datasets/features.parquet


## Teste rápido: as features batem a média histórica?

Usa a mesma divisão temporal do trabalho (teste: 2º semestre/2025 a 1º semestre/2026). O alvo aqui é a **contagem** `n_furtos`; para o seu índice `risco_furto`, faça o merge pela chave (`h3`, `DATA`) e troque `ALVO_AVAL`.

É uma checagem barata (amostra + poucas árvores), não substitui sua busca de hiperparâmetros.

In [11]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

ALVO_AVAL = TARGET                      # troque pelo seu alvo (ex.: "risco_furto" após merge)
CORTE_TESTE = pd.Timestamp("2025-07-01")
MIN_HIST_TREINO = 12                    # 0, 12 ou 24 (veja a explicação acima)
N_AMOSTRA = 400_000

treino = df_final[(df_final[COL_DATA] < CORTE_TESTE) & (df_final["n_hist"] >= MIN_HIST_TREINO)]
teste = df_final[df_final[COL_DATA] >= CORTE_TESTE]
print(f"treino: {len(treino):,} | teste: {len(teste):,}")


def metricas(y, yhat):
    return {"MAE": mean_absolute_error(y, yhat),
            "RMSE": float(np.sqrt(mean_squared_error(y, yhat)))}


# Baselines sem treino
res = {}
for nome in [f"{p_alvo}_exp_mean", f"{p_alvo}_mean12", f"{p_alvo}_saz_mean", f"{p_alvo}_ewm6"]:
    res[f"baseline: {nome}"] = metricas(teste[ALVO_AVAL], teste[nome].fillna(0))

# Conjuntos de features
cal = ["mes", "quinzena", "qz_ano", "qz_sin", "qz_cos"]
proprio = [f for f in FEATURES if f.startswith(p_alvo + "_")]
viz = [f for f in FEATURES if f.startswith("viz") and f.split("_", 1)[1].startswith(p_alvo + "_")]

CONJUNTOS = {
    "RF: calendário": cal,
    "RF: calendário + própria célula": cal + proprio,
    "RF: + vizinhança": cal + proprio + viz,
    "RF: tudo": FEATURES,   # inclui as séries de roubo/total e o nível global
}

am = treino.sample(min(N_AMOSTRA, len(treino)), random_state=42)
modelos = {}
for nome, cols in CONJUNTOS.items():
    rf = RandomForestRegressor(n_estimators=100, min_samples_leaf=20, max_features=0.33,
                               max_samples=0.5, n_jobs=-1, random_state=42)
    rf.fit(am[cols].fillna(-1), am[ALVO_AVAL])
    res[nome] = metricas(teste[ALVO_AVAL], rf.predict(teste[cols].fillna(-1)))
    modelos[nome] = (rf, cols)

pd.DataFrame(res).T.sort_values("RMSE")

treino: 91,296 | teste: 30,432


,MAE,RMSE
baseline: furtos_ewm6,2.537756,14.735367
baseline: furtos_mean12,2.665719,18.033842
RF: tudo,2.681161,19.455893
RF: + vizinhança,2.687320,20.349693
RF: calendário + própria célula,2.697957,20.700346
baseline: furtos_exp_mean,3.079176,25.460703
baseline: furtos_saz_mean,3.191323,25.985617
RF: calendário,18.281985,72.412539


In [12]:
rf, cols = modelos["RF: tudo"]
imp = pd.Series(rf.feature_importances_, index=cols).sort_values(ascending=False)
imp.head(20)

total_lag6        0.053323
furtos_mean24     0.049263
total_lag4        0.043493
furtos_mean12     0.038432
total_mean12      0.038107
roubos_mean2      0.035578
roubos_lag6       0.034914
total_lag12       0.034808
total_ewm12       0.028883
total_ewm6        0.026872
roubos_ewm6       0.026757
total_max12       0.026708
roubos_lag4       0.026588
roubos_lag2       0.026582
furtos_ewm2       0.022307
furtos_ewm12      0.021984
total_mean24      0.021020
furtos_mean4      0.020828
total_exp_mean    0.020811
furtos_mean2      0.019902
dtype: float64

## Próximos passos

- Se "RF: tudo" não superar `baseline: exp_mean`, o problema provavelmente é o alvo (muito esparso/ruidoso nessa resolução) e não as features; vale testar quinzenas agregadas em janelas maiores ou H3 resolução 6.
- Para o modelo de roubo, troque `TARGET = "n_roubos"` e rode de novo (as features já existem para as duas séries). Para usar no seu pipeline, carregue `features_furto.parquet`, filtre `n_hist >= MIN_HIST_TREINO` e use `features_lista.json`. No `get_folds`, as features já são só passado, então `gap=0` está seguro.
- Features das outras séries (`roubos_*`, `total_*`, `global_*`) podem ser removidas se aumentarem demais o tempo de treino; olhe a importância antes.